<a href="https://colab.research.google.com/github/Andres10-cmd/aiml-pro1/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import os

# ==========================================
# STEP 1: Load and Preprocess the Graph CSV
# ==========================================
# Check if 'graph_dataset.csv' exists; if not, create a mock dataset for testing
if not os.path.exists('graph_dataset.csv'):
    print("'graph_dataset.csv' not found. Creating a mock dataset for demonstration...")
    mock_data = {
        'Source': ['A', 'A', 'B', 'C', 'C', 'D', 'E', 'B', 'F', 'A'],
        'Destination': ['B', 'C', 'D', 'A', 'E', 'F', 'F', 'E', 'C', 'F'],
        'Weight': [2.5, 0.0, 1.2, 0.0, 3.4, 1.1, 0.0, 4.5, 2.2, 0.0]
    }
    df_mock = pd.DataFrame(mock_data)
    df_mock.to_csv('graph_dataset.csv', index=False)

df = pd.read_csv('graph_dataset.csv')
print("--- Loaded Graph Dataset ---")
print(df.head(), "\n")

# Encode categorical node labels (e.g., 'A', 'B', 'C') into numeric values
le = LabelEncoder()
all_nodes = list(set(df['Source']).union(set(df['Destination'])))
le.fit(all_nodes)

# Features (X): Source and Destination node indices
X = np.column_stack((
    le.transform(df['Source']),
    le.transform(df['Destination'])
))

# Target (y): Binary target based on weights or edge properties
y = (df['Weight'] > 0).astype(int).values.reshape(-1, 1)

print(f"Processed Features (X) Shape: {X.shape}")
print(f"Processed Target (y) Shape: {y.shape}\n")


# ==========================================
# STEP 2: Define the Neural Network Class
# ==========================================
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.lr = learning_rate

        # Initialize weights and biases using He/Xavier-like scaling
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def sigmoid_derivative(self, sig):
        return sig * (1 - sig)

    def forward(self, X):
        # Forward pass
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output):
        m = X.shape[0]

        # Backpropagation algorithm
        error = output - y

        # Output layer gradients
        d_W2 = np.dot(self.a1.T, error * self.sigmoid_derivative(output)) / m
        d_b2 = np.sum(error * self.sigmoid_derivative(output), axis=0, keepdims=True) / m

        # Hidden layer gradients
        error_hidden = np.dot(error * self.sigmoid_derivative(output), self.W2.T)
        d_W1 = np.dot(X.T, error_hidden * self.sigmoid_derivative(self.a1)) / m
        d_b1 = np.sum(error_hidden * self.sigmoid_derivative(self.a1), axis=0, keepdims=True) / m

        # Gradient descent weight updates
        self.W1 -= self.lr * d_W1
        self.b1 -= self.lr * d_b1
        self.W2 -= self.lr * d_W2
        self.b2 -= self.lr * d_b2

    def train(self, X, y, epochs=1000):
        loss_history = []
        for epoch in range(epochs):
            output = self.forward(X)
            loss = np.mean(np.square(output - y)) # Mean Squared Error Loss
            loss_history.append(loss)

            self.backward(X, y, output)

            if (epoch + 1) % 200 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.6f}")

        return loss_history


# ==========================================
# STEP 3: Train and Evaluate the Model
# ==========================================
print("--- Training Neural Network with Backpropagation ---")
input_dim = X.shape[1]
hidden_dim = 4
output_dim = 1

nn = NeuralNetwork(input_dim, hidden_dim, output_dim, learning_rate=0.5)
loss_history = nn.train(X, y, epochs=1000)

# Final Predictions
final_preds = nn.forward(X)
print("\n--- Training Complete ---")
print("Target Values:\n", y.flatten())
print("Predicted Outputs:\n", np.round(final_preds.flatten(), 4))

'graph_dataset.csv' not found. Creating a mock dataset for demonstration...
--- Loaded Graph Dataset ---
  Source Destination  Weight
0      A           B     2.5
1      A           C     0.0
2      B           D     1.2
3      C           A     0.0
4      C           E     3.4 

Processed Features (X) Shape: (10, 2)
Processed Target (y) Shape: (10, 1)

--- Training Neural Network with Backpropagation ---
Epoch 200/1000 - Loss: 0.223108
Epoch 400/1000 - Loss: 0.216770
Epoch 600/1000 - Loss: 0.210650
Epoch 800/1000 - Loss: 0.205160
Epoch 1000/1000 - Loss: 0.200666

--- Training Complete ---
Target Values:
 [1 0 1 0 1 1 0 1 1 0]
Predicted Outputs:
 [0.6002 0.6242 0.6315 0.2385 0.6309 0.6295 0.6414 0.6216 0.7505 0.6114]
